In [ ]:
# --- SPACEMAP notebook configuration ------------------------------------
# Every path below is built from DATA_DIR. Point it at your own data by
# setting SPACEMAP_DATA_DIR in your environment, or by editing the default.
import os
from pathlib import Path

DATA_DIR = os.environ.get("SPACEMAP_DATA_DIR", str(Path.home() / "spacemap-data"))
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
print(f"Using DATA_DIR = {DATA_DIR}")


# Step 6A: Leiden Clustering Phenotyping with Advanced Subclustering

This notebook performs cell phenotyping using Leiden community detection on batch-corrected marker data. It provides an alternative clustering approach to SOM with identical structure, annotations, and outputs.

## New Feature: Advanced Subclustering with Flexible Filtering

**Advanced Filtering System**: This notebook now supports sophisticated filtering for subclustering with multiple conditions and OR logic.

### Filtering Configuration Examples

#### Example 1: Simple Single-Value Filters
```python
SUBCLUSTERING_FILTER = {
    'phenotype_ml_mr': 'Epithelia',  # Only epithelial cells
    'corrected': 1                   # That are also corrected
}
```

#### Example 2: Multiple Values for Same Column
```python
SUBCLUSTERING_FILTER = {
    'phenotype_ml_mr': ['Epithelia', 'Helper T', 'Treg']  # Multiple phenotypes
}
```

#### Example 3: OR Conditions (Multiple Cell Types)
```python
SUBCLUSTERING_FILTER = {
    'OR_CONDITIONS': [
        {'tumor': 1},           # All tumor cells
        {'epithelia': 1},       # OR all epithelia cells
        {'phenotype_ml_mr': ['Helper T', 'Treg']}  # OR specific immune cells
    ]
}
```

#### Example 4: Complex Mixed Conditions
```python
SUBCLUSTERING_FILTER = {
    'OR_CONDITIONS': [
        {'tumor': 1},                                       # Tumor cells
        {'epithelia': 1},                                   # OR epithelia cells  
        {'phenotype_ml_mr': ['Epithelia', 'Helper T', 'Treg']}  # OR specific phenotypes
    ],
    'corrected': 1,  # AND all must be corrected (applies to ALL above)
    'tissue': 1      # AND all must be tissue regions
}
```

### Custom Output Configuration
```python
CLUSTER_COLUMN_NAME = 'mixed_subcluster'    # Custom cluster column name
PHENOTYPE_COLUMN_NAME = 'mixed_phenotype'   # Custom phenotype column name

SUBCLUSTERING_CORR_MARKERS = [
    "E-cadherin", "Pan-Cytokeratin",  # Epithelial markers
    "CD3e", "CD4", "CD8", "FoxP3",    # T-cell markers  
    "CD31", "Vimentin"                # Additional markers
]
```

## Overview

The Leiden clustering approach:
1. **Data Preparation** - Load batch-corrected marker data with advanced filtering
2. **Advanced Filtering** - Support OR/AND logic and multiple values per column
3. **Normalization** - Shared normalization cache with SOM for consistency
4. **Leiden Clustering** - PCA → neighbors → Leiden → UMAP embedding
5. **Cluster Characterization** - Generate expression heatmaps and statistics
6. **Manual annotation** - Interactive tools for phenotype assignment
7. **Quality Control** - Comprehensive validation and outputs with custom column names

## Input
- Batch-corrected CSV files from Step 5 (uses *_corr markers)
- Optional: phenotype mapping for automated assignment
- **New**: Advanced filtering criteria supporting complex cell selection

## Output
- CSV files with Leiden cluster assignments (custom column names supported)
- CSV files with phenotype annotations (custom column names supported)
- Expression heatmaps for manual annotation
- UMAP visualizations for validation

## Key Features
- **Intelligent clustering** - Geometric sketching for large datasets
- **Advanced filtering** - OR/AND logic with multiple values per column
- **Mixed cell analysis** - Combine different cell types in single analysis
- **Custom output columns** - Configurable cluster and phenotype column names
- **Flexible markers** - Use custom marker sets for mixed populations
- **Shared normalization** - Compatible cache with SOM method
- **Comprehensive outputs** - Multiple visualization methods for validation
- **Interactive annotation** - Professional GUI for phenotype assignment

In [ ]:
# Imports & Environment Setup
import os, sys, warnings, logging
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
warnings.filterwarnings('ignore')

# Add src to path for imports
project_root = Path().resolve().parent
src_path = project_root / "src"
sys.path.insert(0, str(src_path))

from spacemap.config import default_config
from spacemap.ph_leiden_utils import (
    apply_flexible_normalization,
    load_or_compute_normalized,
    characterize_clusters,
    create_cluster_heatmap,
    launch_phenotype_assignment_app,
    apply_phenotypes,
    intelligent_leiden_clustering
)

try:
    import scanpy as sc
    print(f'Scanpy {sc.__version__} available')
except Exception:
    print('Scanpy not available; install to run Leiden.')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('step6a_leiden')

print(f"Project root: {project_root}")
print("Step 6A: Leiden Phenotyping Setup Complete")

## Feature Variability Discovery (Optional)

**Purpose**: Discover the most variable markers within specific cell populations to guide marker selection for clustering.

This analysis helps you:
1. Identify which markers show the most variation within your population of interest
2. Select informative markers for downstream clustering
3. Compare variability across different cell types or conditions

**Workflow**:
- Define filtering criteria (similar to subclustering filters)
- Specify which groups to analyze separately
- Select candidate markers to test
- Generate variability tables and plots for each group

**When to use**: Run this before configuring your clustering to identify the most discriminative markers.

In [ ]:
# =============================================================================
# INSPECT COLUMN VALUES FOR VARIABILITY GROUPING
# =============================================================================

# Specify which column to inspect
INSPECT_COLUMN = 'phenotype_ml_consensus_refined'

# Load data to inspect available values
inspect_file = Path(f'{DATA_DIR}/example_study/registration/roi/cellpose/annotated_csv/corrected/batch_corrected_merged.csv')

if inspect_file.exists():
    inspect_df = pd.read_csv(inspect_file, low_memory=False)
    
    if INSPECT_COLUMN in inspect_df.columns:
        unique_values = inspect_df[INSPECT_COLUMN].value_counts().sort_values(ascending=False)
        
        print(f"Unique values in column '{INSPECT_COLUMN}':")
        print("="*80)
        print(f"{'Value':<40} {'Count':>15} {'Percentage':>10}")
        print("-"*80)
        
        for value, count in unique_values.items():
            percentage = (count / len(inspect_df)) * 100
            print(f"{str(value):<40} {count:>15,} {percentage:>9.2f}%")
        
        print("="*80)
        print(f"Total unique values: {len(unique_values)}")
        print(f"Total cells: {len(inspect_df):,}")
        
        # Show example list format for configuration
        print(f"\nExample configuration for VARIABILITY_GROUP_VALUES:")
        print(f"VARIABILITY_GROUP_VALUES = {list(unique_values.head(10).index)}")
        
    else:
        print(f"Column '{INSPECT_COLUMN}' not found in data.")
        print(f"Available columns: {list(inspect_df.columns)}")
else:
    print(f"Data file not found: {inspect_file}")

In [ ]:
# =============================================================================
# FEATURE VARIABILITY DISCOVERY CONFIGURATION
# =============================================================================

# Set to True to run variability discovery, False to skip
RUN_VARIABILITY_DISCOVERY = True

if RUN_VARIABILITY_DISCOVERY:
    from spacemap.ph_leiden_utils import discover_variable_features, plot_variability_results, plot_combined_variability_comparison
    
    # Input file
    VARIABILITY_INPUT_FILE = project_root / 'outputs/phenotyping/all_samples_combined_corrected_phenotyped.csv'
    VARIABILITY_OUTPUT_DIR = project_root / 'outputs/phenotyping/variability_discovery'
    
    # =============================================================================
    # FILTERING CONFIGURATION FOR VARIABILITY ANALYSIS
    # =============================================================================
    
    # Base filtering criteria (AND conditions applied to ALL groups)
    # These filters are applied BEFORE splitting by group
    VARIABILITY_BASE_FILTERS = {
        #'corrected': 1,  # Only analyze corrected cells
        'tissue': 1,   # Optional: only tissue regions
    }
    
    # =============================================================================
    # GROUP ANALYSIS CONFIGURATION
    # =============================================================================
    
    # Column to split analysis by (each unique value analyzed separately)
    VARIABILITY_GROUP_COLUMN = 'phenotype_ml_mr'
    
    # List of values in that column to analyze
    # Examples:
    # VARIABILITY_GROUP_VALUES = ['Helper T', 'Cytotoxic T', 'Epithelia', 'Macrophages']
    # VARIABILITY_GROUP_VALUES = ['Tumor', 'Stroma', 'Immune']
    VARIABILITY_GROUP_VALUES = ['Epithelia', 'CAF', 'Granulocyte', 'Mac', 'Plasma B cell', 'Endothelia', 'Helper T', 'Cytotoxic T', 'DC', 'Treg'
        
    ]
    
    # Alternatively, use a different column for grouping:
    # VARIABILITY_GROUP_COLUMN = 'tumor'
    # VARIABILITY_GROUP_VALUES = [0, 1]  # Compare tumor vs non-tumor
    
    # =============================================================================
    # MARKER SELECTION FOR VARIABILITY TESTING
    # =============================================================================
    
    # Corrected markers to test (will add _corr suffix automatically)
    VARIABILITY_CORR_MARKERS = [
        'DAPI', 'CD44', 'CD31', 'CD107a', 'E-cadherin', 'CD20', 'CD4', 'CD45RO', 'Pan-Cytokeratin', 'CD68',
        'Podoplanin', 'HLA-A', 'CD11c', 'CD8', 'HLA-DR', 'Ki67', 'CD3e', 'CD69', 'CD45', 'CD15', 'gH2AX',
        'FoxP3', 'CD141', 'PDGFRa', 'aSMA', 'CD163', 'FN1', 'S100A9', 'CD206', 'CD11b', 'CD38',
        'PD-L1', 'Vimentin', 'CD40', 'PCNA'
    ]
    
    # Non-corrected markers to test (no suffix added)
    VARIABILITY_NON_CORR_MARKERS = [
        'area', 
        'major_axis_length', 
        'minor_axis_length',
        'eccentricity',
        'solidity'
    ]
    
    # Normalization method for variability calculation
    # Options: 'arcsinh', 'zscore', 'none'
    VARIABILITY_NORMALIZATION = 'arcsinh'
    VARIABILITY_COFACTOR = 5
    
    # Plotting parameters
    TOP_N_MARKERS = 20  # Number of top markers to show in plots
    
    print("Variability Discovery Configuration:")
    print(f" Input file: {VARIABILITY_INPUT_FILE}")
    print(f" Output directory: {VARIABILITY_OUTPUT_DIR}")
    print(f" Base filters: {VARIABILITY_BASE_FILTERS}")
    print(f" Grouping by: {VARIABILITY_GROUP_COLUMN}")
    print(f" Groups to analyze: {VARIABILITY_GROUP_VALUES}")
    print(f" Testing {len(VARIABILITY_CORR_MARKERS)} corrected + {len(VARIABILITY_NON_CORR_MARKERS)} non-corrected markers")
else:
    print("Variability discovery DISABLED. Set RUN_VARIABILITY_DISCOVERY = True to enable.")

In [ ]:
# =============================================================================
# RUN FEATURE VARIABILITY DISCOVERY
# =============================================================================

if RUN_VARIABILITY_DISCOVERY:
    print("="*80)
    print("FEATURE VARIABILITY DISCOVERY")
    print("="*80)
    
    # Load data
    if not VARIABILITY_INPUT_FILE.exists():
        raise FileNotFoundError(f"Input file not found: {VARIABILITY_INPUT_FILE}")
    
    var_data = pd.read_csv(VARIABILITY_INPUT_FILE, low_memory=False)
    print(f"\nLoaded data: {len(var_data):,} cells")
    
    # Run variability analysis
    variability_results = discover_variable_features(
        df=var_data,
        filter_criteria=VARIABILITY_BASE_FILTERS,
        group_column=VARIABILITY_GROUP_COLUMN,
        group_values=VARIABILITY_GROUP_VALUES,
        corr_markers=VARIABILITY_CORR_MARKERS,
        non_corr_markers=VARIABILITY_NON_CORR_MARKERS,
        output_dir=VARIABILITY_OUTPUT_DIR,
        normalization_method=VARIABILITY_NORMALIZATION,
        cofactor=VARIABILITY_COFACTOR
    )
    
    print(f"\n{'='*80}")
    print(f"VARIABILITY ANALYSIS COMPLETE")
    print(f"Analyzed {len(variability_results)} groups")
    print(f"Results saved to: {VARIABILITY_OUTPUT_DIR}")
    print(f"{'='*80}")
    
    # Generate individual plots for each group
    print("\nGenerating individual plots for each group...")
    plot_variability_results(
        variability_results=variability_results,
        output_dir=VARIABILITY_OUTPUT_DIR,
        top_n=TOP_N_MARKERS
    )
    
    # Generate comparison plots if multiple groups
    if len(variability_results) > 1:
        print("\nGenerating comparison plots across groups...")
        plot_combined_variability_comparison(
            variability_results=variability_results,
            output_dir=VARIABILITY_OUTPUT_DIR,
            top_n=15
        )
    
    # Display summary table for all groups
    print(f"\n{'='*80}")
    print("SUMMARY: Top Variable Markers by Group")
    print(f"{'='*80}\n")
    
    for group_name, var_df in variability_results.items():
        print(f"\n{group_name} (n={var_df['n_cells'].iloc[0]:,} cells):")
        print("-" * 60)
        top_10 = var_df[['marker', 'std_dev']].head(10)
        for idx, row in top_10.iterrows():
            print(f"  {row['marker']:30s} {row['std_dev']:.4f}")
    
    print(f"\n{'='*80}")
    print("NEXT STEPS:")
    print("1. Review the generated plots in:", VARIABILITY_OUTPUT_DIR / 'variability_plots')
    print("2. Check the CSV files for complete rankings")
    print("3. Select high-variability markers for your CORR_MARKERS configuration")
    print("4. Re-run clustering with optimized marker selection")
    print(f"{'='*80}\n")
    
    # Display one plot as example
    from IPython.display import Image, display
    
    # Show first group's bar plot
    first_group = list(variability_results.keys())[0]
    safe_name = str(first_group).replace(' ', '_').replace('/', '_')
    example_plot = VARIABILITY_OUTPUT_DIR / 'variability_plots' / f'variability_bar_{safe_name}.png'
    
    if example_plot.exists():
        print(f"\nExample plot for: {first_group}")
        display(Image(filename=str(example_plot)))
    
    # Show comparison heatmap if available
    heatmap_path = VARIABILITY_OUTPUT_DIR / 'variability_plots' / 'variability_comparison_heatmap.png'
    if heatmap_path.exists():
        print(f"\nComparison heatmap across all groups:")
        display(Image(filename=str(heatmap_path)))

else:
    print("Variability discovery skipped. Set RUN_VARIABILITY_DISCOVERY = True to run.")

## Configuration

Configure the Leiden clustering parameters below. These settings control the clustering resolution, neighborhood size, and visualization options.

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# I/O paths
INPUT_DIR = project_root / 'outputs/phenotyping'
OUTPUT_DIR = project_root / 'outputs/phenotyping'
PLOTS_DIR = OUTPUT_DIR / 'leiden/plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
input_file = INPUT_DIR / 'all_samples_combined_corrected_phenotyped.csv'

# Normalization flags to match SOM method exactly
NORMALIZATION_FLAGS = dict(z1=True, z2=False, cdf=False, nlog=False)
NORMALIZATION_GROUP_COLS = ['batch_id']

# =============================================================================
# FILTERING CONFIGURATION
# =============================================================================

# Advanced Filtering System: Define which cells to analyze
# SUPPORTS MULTIPLE FILTER TYPES:

# TYPE 1: Simple single-value filters
# FILTER_CRITERIA = {'phenotype_ml_mr': 'Epithelia', 'corrected': 1}

# TYPE 2: Multiple values for same column (use list)
# FILTER_CRITERIA = {'phenotype_ml_mr': ['Epithelia', 'Helper T', 'Treg']}

# TYPE 3: Multiple conditions with OR logic (use 'OR_CONDITIONS' key)
# FILTER_CRITERIA = {
#     'OR_CONDITIONS': [
#         {'tumor': 1},           # Cells where tumor == 1
#         {'epithelia': 1},       # OR cells where epithelia == 1
#         {'phenotype_ml_mr': ['Epithelia', 'Helper T']}  # OR cells with multiple phenotypes
#     ]
# }

# TYPE 4: Complex mixed conditions (OR + AND logic)
FILTER_CRITERIA = {
    'OR_CONDITIONS': [
        #{'tumor': 1},                                        # All tumor cells
        #{'epithelia': 1},                                    # OR all epithelia cells  
        {'phenotype_ml_mr': ['Epithelia']}  # OR specific phenotypes ['Epithelia', 'Helper T', 'Treg']
    ],
    'corrected': 1,  # AND must be corrected (applies to ALL above conditions)
}

# For standard clustering (no subclustering), use simple filters:
# FILTER_CRITERIA = {
#     'tissue': 1,          # Filter to cells where tissue == 1 (use None to disable)
#     'corrected': 1,       # Example: could also filter by corrected == 1
# }

MIN_MARKER_SUM = 0.1      # Filter out cells with low total expression

# =============================================================================
# OUTPUT CONFIGURATION
# =============================================================================

# Custom output column names (change these to create custom cluster/phenotype columns)
CLUSTER_COLUMN_NAME = 'leiden_cluster_epithelia'        # Custom cluster column name
PHENOTYPE_COLUMN_NAME = 'epithelia_phenotype'           # Custom phenotype column name

# For standard clustering, use default names:
# CLUSTER_COLUMN_NAME = 'leiden_cluster'        
# PHENOTYPE_COLUMN_NAME = 'phenotype_leiden'   

# =============================================================================
# MARKER CONFIGURATION
# =============================================================================

# Column Configuration
CELL_ID_COLUMN = 'cell_id'     # Name of the cell ID column (configurable)

# Corrected markers (used for all clustering - standard and subclustering)
CORR_MARKERS = [    
    'Ki67', 'CD44', 'gH2AX', 'PD-L1', 'E-cadherin', 
    'PCNA', 'Vimentin', 'Pan-Cytokeratin'
]

# Non-corrected markers (no _corr suffix)
NON_CORR_MARKERS = [#'area'
]

# Combine markers for clustering
MARKERS_BASE = CORR_MARKERS + NON_CORR_MARKERS
CLUSTERING_MARKERS_CORR = [f"{m}_corr" for m in CORR_MARKERS] + NON_CORR_MARKERS

# =============================================================================
# CLUSTERING PARAMETERS
# =============================================================================

# Leiden clustering parameters - Optimized for meaningful biological clusters
LEIDEN_RESOLUTION = 0.5        # Higher resolution for finer cluster separation
LEIDEN_N_NEIGHBORS = 15        # Neighborhood size for graph construction
LEIDEN_N_PCS = 21              # Principal components for dimensionality reduction
LEIDEN_N_CLUSTERS_TARGET = 15  # Target number of meaningful clusters
SUBSAMPLE_SIZE = 150000        # Subsample size for computational efficiency

# Processing options
USE_INTELLIGENT_CLUSTERING = True  # Use geometric sketching for large datasets
GENERATE_PLOTS = True             # Generate visualization plots
RANDOM_SEED = 42                  # Reproducibility seed

# =============================================================================
# CONFIGURATION SUMMARY
# =============================================================================

# Determine clustering mode based on filter complexity
IS_SUBCLUSTERING = ('OR_CONDITIONS' in FILTER_CRITERIA or 
                   any(isinstance(v, list) for v in FILTER_CRITERIA.values() if v is not None) or
                   any(k in ['phenotype_ml_mr', 'tumor', 'epithelia'] for k in FILTER_CRITERIA.keys()))

if IS_SUBCLUSTERING:
    clustering_mode = "SUBCLUSTERING"
    print("SUBCLUSTERING MODE DETECTED")
    print(f"Filter criteria: {FILTER_CRITERIA}")
    print(f"Custom output columns: {CLUSTER_COLUMN_NAME}, {PHENOTYPE_COLUMN_NAME}")
else:
    clustering_mode = "STANDARD"
    print("STANDARD CLUSTERING MODE")
    print(f"Filter criteria: {FILTER_CRITERIA}")
    print(f"Output columns: {CLUSTER_COLUMN_NAME}, {PHENOTYPE_COLUMN_NAME}")

print("Configuration loaded:")
print(f" Input dir: {INPUT_DIR}")
print(f" Output plots: {PLOTS_DIR}")
print(f" Cell ID column: {CELL_ID_COLUMN}")
print(f" Clustering mode: {clustering_mode}")

print(f" Selected markers: {len(MARKERS_BASE)} total")
print(f" Corrected markers: {len([m for m in CLUSTERING_MARKERS_CORR if m.endswith('_corr')])}")
print(f" Non-corrected markers: {len([m for m in CLUSTERING_MARKERS_CORR if not m.endswith('_corr')])}")
print(f" Normalization: {NORMALIZATION_FLAGS}")
print(f" Intelligent clustering: {USE_INTELLIGENT_CLUSTERING}")
print(f" Target clusters: {LEIDEN_N_CLUSTERS_TARGET}")
print(f" Leiden resolution: {LEIDEN_RESOLUTION}")

## Step 1: Data Loading and Preparation


Load the batch-corrected data and prepare it for Leiden clustering analysis.- Results in a cleaner, more consistent dataset for phenotyping

- Uses configurable cell ID column for proper anchoring

**Flexible filtering system**:- Can add/remove filters or disable with `None` values

- Configurable filtering via `FILTER_CRITERIA` dictionary- Default: only cells with `tissue=1` (tissue regions)

In [ ]:
# =============================================================================
# STEP 1: DATA LOADING & NORMALIZATION (SHARED CACHE WITH SOM)
# =============================================================================
print("Step 1: Loading data & preparing Leiden input...")

if not input_file.exists():
    raise FileNotFoundError(f"Combined corrected data not found: {input_file}")

data = pd.read_csv(input_file, low_memory=False)
print(f"Loaded: {len(data):,} rows")

# =============================================================================
# ADVANCED FILTERING SYSTEM
# =============================================================================

def apply_advanced_filter(df, filter_criteria):
    """
    Apply advanced filtering that supports:
    1. Simple filters: {'column': value}
    2. Multiple values: {'column': [value1, value2, value3]}
    3. OR conditions: {'OR_CONDITIONS': [filter1, filter2, filter3]}
    4. Mixed conditions: OR conditions + AND conditions
    """
    
    if not filter_criteria:
        return df
    
    print(f"\nApplying advanced filtering...")
    original_count = len(df)
    
    # Separate OR conditions from regular AND conditions
    or_conditions = filter_criteria.pop('OR_CONDITIONS', None)
    and_conditions = {k: v for k, v in filter_criteria.items() if v is not None}
    
    result_df = df.copy()
    
    # Apply OR conditions first (if any)
    if or_conditions:
        print(f" Processing {len(or_conditions)} OR conditions...")
        or_masks = []
        
        for i, condition in enumerate(or_conditions):
            condition_mask = pd.Series([True] * len(result_df), index=result_df.index)
            condition_desc = []
            
            for column, value in condition.items():
                if column not in result_df.columns:
                    print(f"   Warning: '{column}' not found - skipping OR condition {i+1}")
                    condition_mask = pd.Series([False] * len(result_df), index=result_df.index)
                    break
                
                if isinstance(value, list):
                    # Multiple values for same column
                    col_mask = result_df[column].isin(value)
                    condition_desc.append(f"{column} in {value}")
                else:
                    # Single value
                    col_mask = result_df[column] == value
                    condition_desc.append(f"{column}=={value}")
                
                condition_mask = condition_mask & col_mask
            
            or_masks.append(condition_mask)
            cells_matching = condition_mask.sum()
            print(f"   OR condition {i+1} ({', '.join(condition_desc)}): {cells_matching:,} cells")
        
        # Combine all OR conditions
        if or_masks:
            combined_or_mask = or_masks[0]
            for mask in or_masks[1:]:
                combined_or_mask = combined_or_mask | mask
            
            result_df = result_df[combined_or_mask].copy()
            or_filtered_count = len(result_df)
            print(f" OR filtering result: {or_filtered_count:,}/{original_count:,} cells retained")
        else:
            print(f" No valid OR conditions found")
    
    # Apply AND conditions
    if and_conditions:
        print(f" Processing {len(and_conditions)} AND conditions...")
        
        for column, value in and_conditions.items():
            if column not in result_df.columns:
                print(f"   Warning: '{column}' not found - skipping AND condition")
                continue
            
            before_count = len(result_df)
            
            if isinstance(value, list):
                # Multiple values for same column  
                result_df = result_df[result_df[column].isin(value)].copy()
                print(f"   AND condition {column} in {value}: {len(result_df):,}/{before_count:,}")
            else:
                # Single value
                result_df = result_df[result_df[column] == value].copy()
                print(f"   AND condition {column}=={value}: {len(result_df):,}/{before_count:,}")
    
    final_count = len(result_df)
    print(f"\nFinal filtering result: {final_count:,}/{original_count:,} cells ({final_count/original_count*100:.1f}%)")
    
    return result_df

# Apply the advanced filtering using the unified FILTER_CRITERIA
data = apply_advanced_filter(data, FILTER_CRITERIA.copy())

# Verify the configurable cell ID column exists for proper anchoring
if CELL_ID_COLUMN not in data.columns:
    raise ValueError(f"Critical: '{CELL_ID_COLUMN}' column missing from input data. Cannot anchor cells properly.")

print(f"Cell ID column found: {data[CELL_ID_COLUMN].nunique():,} unique cell_ids out of {len(data):,} rows")

# Keep original dataframe intact to preserve spatial coordinates and cell ID
original_df = data.copy()

available_corr = [c for c in CLUSTERING_MARKERS_CORR if c in data.columns]
if not available_corr:
    raise ValueError('No corrected clustering markers in data')

# Generate cache name based on PHENOTYPE_COLUMN_NAME
# Extract the base name (remove '_phenotype' or '_cluster' suffixes)
base_name = PHENOTYPE_COLUMN_NAME.replace('_phenotype', '').replace('_cluster', '')
cache_name = f'normalized_markers_{base_name}.csv'

CACHE_DIR = OUTPUT_DIR / 'normalized_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
cache_csv = CACHE_DIR / cache_name

# Always delete existing cache to force fresh normalization
if cache_csv.exists():
    cache_csv.unlink()
    print(f"Deleted existing cache: {cache_csv}")

print(f"Using normalization cache: {cache_csv}")
print(f"Cache will be regenerated fresh (no existing cache)")

# Use configurable cell ID column for proper anchoring
norm_df = load_or_compute_normalized(
    df=data[[CELL_ID_COLUMN] + NORMALIZATION_GROUP_COLS + available_corr],
    marker_cols=available_corr,
    group_cols=NORMALIZATION_GROUP_COLS,
    suffix='_norm',
    normalization_flags=NORMALIZATION_FLAGS,
    cache_path=cache_csv,
    id_col=CELL_ID_COLUMN
)

norm_cols = [c for c in norm_df.columns if c.endswith('_norm')]
print(f'Normalized columns: {len(norm_cols)} (cache={cache_csv.exists()})')

# Merge normalized data with original data using cell ID
if CELL_ID_COLUMN in norm_df.columns:
    data = data.merge(norm_df, on=CELL_ID_COLUMN, how='left')
else:
    # Fallback to index-based merge if no cell ID
    norm_df = norm_df.reindex(data.index)
    data = pd.concat([data, norm_df], axis=1)

# Apply expression filtering
pre_filter_count = len(data)
raw_expression_sums = data[available_corr].sum(axis=1)
data = data[raw_expression_sums > MIN_MARKER_SUM]
filtered_count = pre_filter_count - len(data)

print(f"\nExpression filtering:")
print(f" Before expression filter: {pre_filter_count:,} cells")
print(f" After expression filter: {len(data):,} cells")
print(f" Cells removed: {filtered_count:,} ({filtered_count/pre_filter_count*100:.2f}%)")

# Prepare clustering matrix
X = data[norm_cols].fillna(0).astype(float)
print(f"\nData prepared for clustering: {len(data):,} cells x {len(norm_cols)} markers")

# Display clustering information based on mode
if IS_SUBCLUSTERING:
    print(f"\n=== SUBCLUSTERING MODE ===")
    print(f"Successfully filtered {len(data):,} cells using advanced criteria")
    print(f"Using {len(available_corr)} markers for subclustering")
    print(f"Output columns will be: '{CLUSTER_COLUMN_NAME}', '{PHENOTYPE_COLUMN_NAME}'")
else:
    print(f"\n=== STANDARD CLUSTERING MODE ===")
    print(f"Standard clustering of {len(data):,} cells")
    print(f"Using {len(available_corr)} markers")
    print(f"Output columns will be: '{CLUSTER_COLUMN_NAME}', '{PHENOTYPE_COLUMN_NAME}'")

## Step 2: Leiden Clustering

Perform Leiden community detection clustering on the normalized marker data.

**Intelligent clustering approach**:
- **Geometric sketching** - Density-aware subsampling for large datasets
- **Optimized parameters** - Balanced resolution and neighborhood size
- **UMAP embedding** - 2D visualization coordinates for validation
- **Quality control** - Automatic detection of over-clustering artifacts

**Expected output**: 10-20 biologically meaningful clusters with UMAP coordinates for visualization.

In [ ]:
# =============================================================================
# STEP 2: LEIDEN CLUSTERING
# =============================================================================
print("Step 2: Performing Leiden clustering...")

start_time = time.time()

print(f"Dataset: {len(data):,} cells x {len(norm_cols)} markers")
print(f"Parameters: resolution={LEIDEN_RESOLUTION}, neighbors={LEIDEN_N_NEIGHBORS}, PCs={LEIDEN_N_PCS}")
print(f"Target clusters: ~{LEIDEN_N_CLUSTERS_TARGET}")
print(f"Output cluster column: '{CLUSTER_COLUMN_NAME}'")

# Choose clustering approach based on dataset size
if USE_INTELLIGENT_CLUSTERING and len(data) > 50000:
    print("Using intelligent clustering approach (recommended for large datasets)")
    
    # Set up parameters for intelligent clustering
    leiden_params = {
        'n_pcs': LEIDEN_N_PCS,
        'n_neighbors': LEIDEN_N_NEIGHBORS,
        'resolution': LEIDEN_RESOLUTION,
        'subsample_size': SUBSAMPLE_SIZE,
        'random_seed': RANDOM_SEED
    }
    
    # Run intelligent clustering
    cluster_assignments, umap_coords = intelligent_leiden_clustering(
        df=data,
        norm_cols=norm_cols,
        leiden_params=leiden_params,
        logger=logger
    )
    
    # Assign results to dataframe using custom column names
    data[CLUSTER_COLUMN_NAME] = cluster_assignments
    data['umap_x'] = umap_coords[:, 0]
    data['umap_y'] = umap_coords[:, 1]
    
else:
    print("Using direct scanpy approach")
    
    # Create AnnData object for clustering
    obs_cols = ['batch_id', 'tissue_roi', 'x_centroid', 'y_centroid'] 
    obs_meta = data[[col for col in obs_cols if col in data.columns]].copy()
    
    adata = sc.AnnData(X=X.values, obs=obs_meta)
    adata.var_names = norm_cols
    adata.obs_names_make_unique()
    
    # Perform PCA
    max_pcs = min(adata.shape) - 1
    actual_n_pcs = min(LEIDEN_N_PCS, max_pcs)
    sc.tl.pca(adata, n_comps=actual_n_pcs, svd_solver='randomized', random_state=RANDOM_SEED)
    
    # Build neighborhood graph
    sc.pp.neighbors(adata, n_neighbors=LEIDEN_N_NEIGHBORS, n_pcs=actual_n_pcs,
                   use_rep='X_pca', random_state=RANDOM_SEED)
    
    # Run Leiden clustering
    sc.tl.leiden(adata, resolution=LEIDEN_RESOLUTION, random_state=RANDOM_SEED, key_added='leiden')
    
    # Compute UMAP embedding
    sc.tl.umap(adata, random_state=RANDOM_SEED)
    
    # Assign results to main dataframe using custom column names
    data[CLUSTER_COLUMN_NAME] = adata.obs['leiden'].astype(int).values
    data['umap_x'] = adata.obsm['X_umap'][:, 0]
    data['umap_y'] = adata.obsm['X_umap'][:, 1]

n_clusters = data[CLUSTER_COLUMN_NAME].nunique()
total_time = time.time() - start_time

print(f"\nClustering completed in {total_time/60:.1f} minutes")
print(f"Final clusters: {n_clusters}")
print(f"Cells processed: {len(data):,}")

# Quality check
cluster_sizes = data[CLUSTER_COLUMN_NAME].value_counts().sort_index()
print(f"Cluster size distribution: min={cluster_sizes.min()}, max={cluster_sizes.max()}, median={cluster_sizes.median()}")

if n_clusters < 5:
    print("Warning: Very few clusters detected. Consider increasing resolution.")
elif n_clusters > 50:
    print("Warning: Many clusters detected. Consider decreasing resolution.")
else:
    print(f"Cluster count looks reasonable for biological interpretation.")

# Display clustering-specific information
if IS_SUBCLUSTERING:
    print(f"\n=== SUBCLUSTERING RESULTS ===")
    print(f"Generated {n_clusters} subclusters for the filtered cell population")
    print(f"Results saved in column: '{CLUSTER_COLUMN_NAME}'")
else:
    print(f"\n=== STANDARD CLUSTERING RESULTS ===")
    print(f"Generated {n_clusters} clusters for the cell population")
    print(f"Results saved in column: '{CLUSTER_COLUMN_NAME}'")

## Step 3: Cluster Characterization

Compute per-cluster statistics and expression profiles for phenotype assignment guidance.

In [ ]:
# =============================================================================
# STEP 3: CLUSTER CHARACTERIZATION
# =============================================================================
print("Step 3: Characterizing clusters...")

char_df = characterize_clusters(
    data=data,
    cluster_col=CLUSTER_COLUMN_NAME,
    marker_cols=norm_cols
)

# Save characterization with dynamic filename based on column name
if IS_SUBCLUSTERING:
    char_filename = f'{CLUSTER_COLUMN_NAME}_characterization.csv'
else:
    char_filename = 'leiden_cluster_characterization.csv'

char_path = OUTPUT_DIR / char_filename
char_df.to_csv(char_path, index=False)

print(f'Cluster characterization saved: {char_path}')
print(f"\nCharacterized {len(char_df)} clusters with {len(norm_cols)} markers")
char_df.head()

## Step 4: Expression Heatmap

Generate a clustered heatmap showing marker expression patterns across clusters to guide phenotype assignment.

In [ ]:
# =============================================================================
# STEP 4: EXPRESSION HEATMAP
# =============================================================================
print("Step 4: Creating expression heatmap...")

# Dynamic heatmap title and filename based on configuration
if IS_SUBCLUSTERING:
    heatmap_title = f'{CLUSTER_COLUMN_NAME.replace("_", " ").title()} Expression Heatmap'
    heatmap_filename = f'{CLUSTER_COLUMN_NAME}_expression_heatmap.png'
else:
    heatmap_title = 'Leiden Cluster Expression Heatmap'
    heatmap_filename = 'leiden_cluster_expression_heatmap.png'

reordered_cluster_ids, heatmap_file = create_cluster_heatmap(
    char_df=char_df,
    plots_dir=PLOTS_DIR,
    cluster_col=CLUSTER_COLUMN_NAME,
    title=heatmap_title,
    filename=heatmap_filename
)

print(f'Expression heatmap saved: {heatmap_file}')
print(f'Cluster ordering: {reordered_cluster_ids}')

from IPython.display import Image, display
display(Image(filename=str(heatmap_file)))

## Step 5: Interactive Phenotype Assignment

Assign biological phenotypes to clusters using the interactive GUI interface.

**Features**:
- Visual heatmap with clickable cluster selection
- Batch phenotype assignment to selected clusters
- Live assignment tracking with coverage statistics
- Persistent assignments for continued sessions

**Workflow**: Use the GUI to interactively assign phenotypes based on marker expression patterns, or skip to use scripted assignment below.

In [ ]:
# =============================================================================
# STEP 5A: INTERACTIVE PHENOTYPE ASSIGNMENT
# =============================================================================
print("Step 5A: Launching interactive phenotype assignment GUI...")

# Initialize phenotype assignments dictionary
if 'LEIDEN_PHENOTYPES' not in globals():
    LEIDEN_PHENOTYPES = {}

# Ensure the characterization dataframe has the expected column name for the GUI
# The GUI expects either 'som_cluster' or 'leiden_cluster', so we temporarily rename our custom column
char_df_gui = char_df.copy()

# Check if our custom cluster column exists
if CLUSTER_COLUMN_NAME not in char_df_gui.columns:
    raise ValueError(f"Expected cluster column '{CLUSTER_COLUMN_NAME}' not found in characterization dataframe.")

# Create a temporary alias for the GUI (which expects 'leiden_cluster' or 'som_cluster')
char_df_gui = char_df_gui.rename(columns={CLUSTER_COLUMN_NAME: 'leiden_cluster'})

print(f"Using custom cluster column: '{CLUSTER_COLUMN_NAME}' -> 'leiden_cluster' for GUI")

# Launch interactive phenotype assignment interface
LEIDEN_PHENOTYPES = launch_phenotype_assignment_app(
    char_df=char_df_gui,
    reordered_cluster_ids=reordered_cluster_ids,
    meta_cluster_phenotypes=LEIDEN_PHENOTYPES
)

print(f"Phenotype assignments captured: {len(LEIDEN_PHENOTYPES)} clusters")
print(f"Assigned clusters: {list(LEIDEN_PHENOTYPES.keys())}")
print(f"Results will be mapped to column: '{CLUSTER_COLUMN_NAME}'")

## Step 5B: Manual Phenotype Assignment

Alternative scripted approach for reproducible phenotype assignment. Edit the mapping below to assign phenotypes based on cluster analysis.

In [ ]:
# =============================================================================
# 5B) OPTIONAL: MANUAL SCRIPTED PHENOTYPE ASSIGNMENT
# =============================================================================
# Example to edit manually after reviewing heatmap; leave empty lists to skip.
#example_pheno_map = {
    # 'T-Cells': [0, 1, 5],
    # 'B-Cells': [2, 8],
#}
#for pheno, clusters in example_pheno_map.items():
#    for c in clusters:
#        LEIDEN_PHENOTYPES[c] = pheno
#print(f'Total assigned (including scripted): {len(LEIDEN_PHENOTYPES)}')

## Step 6: UMAP Visualization (Optional)

Generate UMAP plots showing cluster and phenotype assignments for quality assessment and validation.

In [ ]:
# =============================================================================
# STEP 6: UMAP VISUALIZATION WITH PHENOTYPES
# =============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Create phenotype labels for visualization using custom column name
if 'LEIDEN_PHENOTYPES' in globals() and LEIDEN_PHENOTYPES:
    data[PHENOTYPE_COLUMN_NAME] = data[CLUSTER_COLUMN_NAME].map(LEIDEN_PHENOTYPES).fillna('Unassigned')
    print(f"Phenotype assignments mapped to {(data[PHENOTYPE_COLUMN_NAME] != 'Unassigned').sum():,} cells")
else:
    data[PHENOTYPE_COLUMN_NAME] = 'Unassigned'
    print("No phenotype assignments found. Using 'Unassigned' for all cells.")

# Check if UMAP coordinates are available
if 'umap_x' not in data.columns or 'umap_y' not in data.columns:
    print("UMAP coordinates not found. Skipping UMAP visualization.")
else:
    # Create subplots for cluster and phenotype coloring
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Subsample for plotting if dataset is too large
    MAX_PLOT_CELLS = 500000
    if len(data) > MAX_PLOT_CELLS:
        plot_idx = np.random.choice(len(data), MAX_PLOT_CELLS, replace=False)
        data_plot = data.iloc[plot_idx].copy()
        print(f"Subsampling {MAX_PLOT_CELLS:,} cells for UMAP plotting")
    else:
        data_plot = data.copy()
        print(f"Plotting all {len(data_plot):,} cells in UMAP")
    
    # Plot 1: Colored by cluster
    scatter1 = axes[0].scatter(data_plot['umap_x'], data_plot['umap_y'], 
                              c=data_plot[CLUSTER_COLUMN_NAME], cmap='tab20', 
                              s=0.5, alpha=0.7, rasterized=True)
    
    # Dynamic title based on clustering mode
    cluster_title = f'UMAP: {CLUSTER_COLUMN_NAME.replace("_", " ").title()} (n={data[CLUSTER_COLUMN_NAME].nunique()})'
    axes[0].set_title(cluster_title)
    axes[0].set_xlabel('UMAP 1')
    axes[0].set_ylabel('UMAP 2')
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Colored by phenotype
    unique_phenotypes = data_plot[PHENOTYPE_COLUMN_NAME].unique()
    color_map = plt.cm.Set3(np.linspace(0, 1, len(unique_phenotypes)))
    phenotype_colors = dict(zip(unique_phenotypes, color_map))
    
    for i, phenotype in enumerate(unique_phenotypes):
        mask = data_plot[PHENOTYPE_COLUMN_NAME] == phenotype
        if mask.sum() > 0:
            axes[1].scatter(data_plot.loc[mask, 'umap_x'], data_plot.loc[mask, 'umap_y'],
                           c=[phenotype_colors[phenotype]], label=phenotype,
                           s=0.5, alpha=0.7, rasterized=True)
    
    # Dynamic title based on phenotype column name
    phenotype_title = f'UMAP: {PHENOTYPE_COLUMN_NAME.replace("_", " ").title()} (n={len(unique_phenotypes)})'
    axes[1].set_title(phenotype_title)
    axes[1].set_xlabel('UMAP 1') 
    axes[1].set_ylabel('UMAP 2')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    
    plt.tight_layout()
    
    # Save the plot with dynamic filename
    if IS_SUBCLUSTERING:
        umap_filename = f'{CLUSTER_COLUMN_NAME}_umap_clusters_phenotypes.png'
    else:
        umap_filename = 'leiden_umap_clusters_phenotypes.png'
    
    umap_plot_path = PLOTS_DIR / umap_filename
    plt.savefig(umap_plot_path, dpi=300, bbox_inches='tight')
    print(f"UMAP plot saved: {umap_plot_path}")
    plt.show()
    
    # Print phenotype summary
    phenotype_counts = data[PHENOTYPE_COLUMN_NAME].value_counts()
    print(f"\nPhenotype distribution:")
    for phenotype, count in phenotype_counts.items():
        print(f"  {phenotype}: {count:,} cells ({count/len(data)*100:.1f}%)")

## Step 7: Apply Phenotypes and Save Results

Apply the phenotype assignments to the dataset and save all outputs in standard format.

**Outputs**: Unified CSV with phenotype assignments, summary statistics, and mapping files compatible with downstream analysis.

In [ ]:
# =============================================================================
# STEP 7: APPLY PHENOTYPES AND SAVE RESULTS
# =============================================================================
print("Step 7: Applying phenotypes and saving results...")

if 'LEIDEN_PHENOTYPES' not in globals() or not LEIDEN_PHENOTYPES:
    print('No phenotype assignments available. Please run Step 5A or 5B first.')
else:
    # Apply phenotypes to the dataset
    # FIXED: Use original data without normalized columns, only add clustering results
    clean_data = data[[c for c in data.columns if not c.endswith('_norm')]].copy()
    
    # Determine method prefix from column names for file naming
    if IS_SUBCLUSTERING:
        # Use custom column name as method prefix, clean it for file naming
        method_prefix = CLUSTER_COLUMN_NAME.replace('_cluster', '').replace('cluster_', '')
        if not method_prefix or method_prefix == 'leiden':
            method_prefix = 'leiden_subcluster'
    else:
        method_prefix = 'leiden'
    
    result = apply_phenotypes(
        data=clean_data,
        cluster_col=CLUSTER_COLUMN_NAME,
        phenotype_map=LEIDEN_PHENOTYPES,
        output_base_dir=OUTPUT_DIR,
        method_prefix=method_prefix,
        cell_id_column=CELL_ID_COLUMN
    )
    
    print(f"\nResults saved successfully:")
    print(f"Main dataset: {result['unified_file']}")
    print(f"Summary statistics: {result['summary_file']}")
    print(f"Phenotype mapping: {result['mapping_file']}")
    
    print(f"\nPhenotype summary:")
    display(result['summary_df'].head(10))
    
    # Display information about custom columns
    if IS_SUBCLUSTERING:
        print(f"\n=== SUBCLUSTERING OUTPUT ===")
        print(f"Custom cluster column: '{CLUSTER_COLUMN_NAME}' added to unified CSV")
        print(f"Custom phenotype column: '{PHENOTYPE_COLUMN_NAME}' added to unified CSV")
        print(f"Generated {data[CLUSTER_COLUMN_NAME].nunique()} subclusters")
    else:
        print(f"\n=== STANDARD CLUSTERING OUTPUT ===")
        print(f"Standard columns added: '{CLUSTER_COLUMN_NAME}', '{PHENOTYPE_COLUMN_NAME}'")

## Step 8: Export Per-Batch CSVs (Optional)

Export individual batch files from the unified phenotyping results for sample-specific analysis.

**Configuration**:
- Set `EXPORT_ALL_BATCHES = True` to export all batches
- Or specify `EXPORT_PREFIXES` list to export selected batch prefixes
- Files saved to `phenotyping/leiden_batch_exports/` directory

In [ ]:
# =============================================================================
# 8) OPTIONAL: EXPORT PER-BATCH CSVs
# =============================================================================
import re
from pathlib import Path

# Configure export behavior here.
EXPORT_ALL_BATCHES = False           # Set True to export every batch_id
EXPORT_PREFIXES = ['P11_roi_1']      # List of prefixes to match (ignored if EXPORT_ALL_BATCHES=True)

# Dynamic export directory naming based on clustering mode
if IS_SUBCLUSTERING:
    export_subdir_name = f'{CLUSTER_COLUMN_NAME.replace("_cluster", "")}_batch_exports'
else:
    export_subdir_name = 'leiden_batch_exports'

EXPORT_SUBDIR = OUTPUT_DIR / export_subdir_name
EXPORT_SUBDIR.mkdir(parents=True, exist_ok=True)

UNIFIED_FILE = OUTPUT_DIR / 'all_samples_combined_corrected_phenotyped.csv'
BATCH_COL = 'batch_id'

# Load unified file (produced after Step 7) if not already in memory
if 'result' in globals() and isinstance(result, dict) and 'unified_file' in result:
    unified_path = Path(result['unified_file'])
else:
    unified_path = UNIFIED_FILE

if not unified_path.exists():
    raise FileNotFoundError(f"Unified phenotyping file not found: {unified_path}\nRun Step 7 first.")

unified_df = pd.read_csv(unified_path)
print(f"Loaded unified phenotyping file: {unified_path} ({len(unified_df):,} rows)")

if BATCH_COL not in unified_df.columns:
    print(f"Column '{BATCH_COL}' not found; skipping export.")
else:
    batches = unified_df[BATCH_COL].dropna().astype(str).unique().tolist()
    if EXPORT_ALL_BATCHES:
        target_batches = batches
        print(f"Exporting ALL {len(target_batches)} batches ...")
    else:
        target_batches = []
        for prefix in EXPORT_PREFIXES:
            matched = [b for b in batches if b.startswith(prefix)]
            target_batches.extend(matched)
        # De-duplicate while preserving order
        seen = set()
        target_batches = [b for b in target_batches if not (b in seen or seen.add(b))]
        print(f"Exporting {len(target_batches)} batches matching prefixes {EXPORT_PREFIXES}")
    
    if not target_batches:
        print('No batches match the export criteria.')
    else:
        for b in target_batches:
            df_b = unified_df[unified_df[BATCH_COL].astype(str) == b].copy()
            safe_name = re.sub(r'[\\/*?:"<>|\s]+', '_', b)
            
            # Dynamic filename based on clustering mode
            if IS_SUBCLUSTERING:
                filename = f'{safe_name}_phenotype_{CLUSTER_COLUMN_NAME.replace("_cluster", "")}.csv'
            else:
                filename = f'{safe_name}_phenotype_leiden.csv'
            
            out_path = EXPORT_SUBDIR / filename
            df_b.to_csv(out_path, index=False)
        
        print(f"Export complete. Files saved under: {EXPORT_SUBDIR}")
        
        # Display information about exported columns
        if IS_SUBCLUSTERING:
            print(f"Exported files contain custom columns: '{CLUSTER_COLUMN_NAME}', '{PHENOTYPE_COLUMN_NAME}'")

## Step 9: Cleanup Normalized Cache

Delete the normalized cache file to ensure fresh computation on next run. This prevents cache inconsistencies and ensures reproducibility.

In [ ]:
# =============================================================================
# STEP 9: CLEANUP NORMALIZED CACHE
# =============================================================================
print("Step 9: Cleaning up normalized cache...")

# Delete the normalization cache after processing is complete
if 'cache_csv' in globals() and cache_csv.exists():
    cache_csv.unlink()
    print(f"Deleted normalized cache: {cache_csv}")
    
    # Also clean up the cache directory if it's empty
    cache_dir = cache_csv.parent
    if cache_dir.exists() and not any(cache_dir.iterdir()):
        cache_dir.rmdir()
        print(f"Removed empty cache directory: {cache_dir}")
else:
    print("No cache file found to clean up")

print("Cache cleanup complete. Fresh normalization will be computed on next run.")

# Display final summary based on clustering mode
if IS_SUBCLUSTERING:
    print(f"\n=== SUBCLUSTERING SUMMARY ===")
    print(f"Successfully subclustered cells using filter: {FILTER_CRITERIA}")
    print(f"Custom output columns: '{CLUSTER_COLUMN_NAME}', '{PHENOTYPE_COLUMN_NAME}'")
    print(f"These columns have been added to the unified CSV file for further analysis.")
else:
    print(f"\n=== STANDARD CLUSTERING SUMMARY ===")
    print(f"Standard Leiden clustering completed.")
    print(f"Output columns: '{CLUSTER_COLUMN_NAME}', '{PHENOTYPE_COLUMN_NAME}'")

## Summary

Leiden clustering workflow completed successfully.

**Key Features**:
- Community detection clustering with optimized parameters
- Automatic dataset size handling and intelligent subsampling
- UMAP embedding for visualization and quality assessment
- Interactive and scripted phenotype assignment options
- Standardized output format compatible with downstream analysis

**Generated Files**:
- `all_samples_combined_corrected_phenotyped.csv` - Main dataset with cluster and phenotype assignments
- `leiden_cluster_characterization.csv` - Detailed cluster statistics and marker profiles
- `leiden_phenotype_summary.csv` - Phenotype distribution and counts
- `leiden_cluster_expression_heatmap.png` - Expression visualization for cluster interpretation

**Cache Management**:
- Normalized cache deleted after completion to ensure fresh computation on next run
- Prevents cache inconsistencies between different runs or parameter changes

**Next Steps**: Proceed to spatial analysis, compare with other clustering methods, or continue with downstream phenotyping workflows.

## Step 10: Merge Subset Phenotypes (Optional)

Create a unified `phenotype_subsets` column that consolidates all subset phenotypes from different cell type subclustering runs. 

**Workflow**:
1. Check for subset phenotype columns (e.g., `phenotype_leiden_epithelia`, `phenotype_leiden_CAF`, etc.)
2. Merge values from all subset columns (prioritize subset phenotypes over base phenotypes)
3. For cells without subset phenotypes, use values from `phenotype_ml_mr`
4. Save updated unified CSV with the new column

**Use case**: This creates a single comprehensive phenotype column that includes both base phenotypes and fine-grained subset phenotypes from subclustering analyses.

In [ ]:
# =============================================================================
# STEP 10: MERGE SUBSET PHENOTYPES INTO UNIFIED COLUMN
# =============================================================================
print("Step 10: Creating unified phenotype_subsets column...")

# Define subset phenotype columns to merge
SUBSET_PHENOTYPE_COLUMNS = [
    'phenotype_leiden_epithelia',
    'phenotype_leiden_CAF',
    'phenotype_leiden_Granulocyte',
    'phenotype_leiden_Mac',
    'phenotype_leiden_Helper',
    'phenotype_leiden_Cytotoxic',
    'phenotype_leiden_DC',
    'phenotype_leiden_Treg'
]

# Fallback column for cells without subset phenotypes
FALLBACK_COLUMN = 'phenotype_ml_mr'

# Load the unified CSV file
UNIFIED_FILE = OUTPUT_DIR / 'all_samples_combined_corrected_phenotyped.csv'

if not UNIFIED_FILE.exists():
    print(f"Warning: Unified file not found at {UNIFIED_FILE}")
    print("Please run Step 7 first to generate the unified CSV file.")
else:
    # Load data
    df = pd.read_csv(UNIFIED_FILE, low_memory=False)
    print(f"Loaded unified file: {len(df):,} rows")
    
    # Check which subset columns exist in the data
    existing_subset_columns = [col for col in SUBSET_PHENOTYPE_COLUMNS if col in df.columns]
    
    if not existing_subset_columns:
        print(f"\nWarning: None of the subset phenotype columns found in data.")
        print(f"Available phenotype columns: {[col for col in df.columns if 'phenotype' in col.lower()]}")
        print("Skipping merge operation.")
    else:
        print(f"\nFound {len(existing_subset_columns)} subset phenotype columns:")
        for col in existing_subset_columns:
            n_values = df[col].notna().sum()
            print(f"  - {col}: {n_values:,} cells with values")
        
        # Initialize the new column with NaN
        df['phenotype_subsets'] = pd.NA
        
        # Merge values from all subset phenotype columns
        # Priority: use the first non-null value found across subset columns
        for col in existing_subset_columns:
            # Fill phenotype_subsets with values from current column where it's still null
            mask = df['phenotype_subsets'].isna() & df[col].notna()
            df.loc[mask, 'phenotype_subsets'] = df.loc[mask, col]
        
        n_from_subsets = df['phenotype_subsets'].notna().sum()
        print(f"\nFilled {n_from_subsets:,} cells from subset phenotype columns")
        
        # Fill remaining cells from fallback column
        if FALLBACK_COLUMN in df.columns:
            mask = df['phenotype_subsets'].isna() & df[FALLBACK_COLUMN].notna()
            df.loc[mask, 'phenotype_subsets'] = df.loc[mask, FALLBACK_COLUMN]
            
            n_from_fallback = mask.sum()
            print(f"Filled {n_from_fallback:,} additional cells from '{FALLBACK_COLUMN}'")
        else:
            print(f"\nWarning: Fallback column '{FALLBACK_COLUMN}' not found in data")
        
        # Summary statistics
        n_total_filled = df['phenotype_subsets'].notna().sum()
        n_blank = df['phenotype_subsets'].isna().sum()
        
        print(f"\n{'='*80}")
        print(f"PHENOTYPE_SUBSETS SUMMARY")
        print(f"{'='*80}")
        print(f"Total cells: {len(df):,}")
        print(f"Cells with phenotype_subsets: {n_total_filled:,} ({n_total_filled/len(df)*100:.2f}%)")
        print(f"  - From subset columns: {n_from_subsets:,}")
        print(f"  - From fallback column: {n_from_fallback:,}")
        print(f"Cells left blank: {n_blank:,} ({n_blank/len(df)*100:.2f}%)")
        
        # Display distribution of phenotypes
        if n_total_filled > 0:
            phenotype_counts = df['phenotype_subsets'].value_counts().head(15)
            print(f"\nTop 15 phenotype_subsets values:")
            print("-" * 80)
            for phenotype, count in phenotype_counts.items():
                print(f"  {str(phenotype):<40} {count:>10,} ({count/n_total_filled*100:>6.2f}%)")
        
        # Save updated unified file
        df.to_csv(UNIFIED_FILE, index=False)
        print(f"\n{'='*80}")
        print(f"Updated unified file saved: {UNIFIED_FILE}")
        print(f"New column 'phenotype_subsets' added successfully")
        print(f"{'='*80}")
        
        # Show example rows
        print(f"\nExample rows with phenotype_subsets:")
        display_cols = ['cell_id', FALLBACK_COLUMN] + existing_subset_columns + ['phenotype_subsets']
        display_cols = [col for col in display_cols if col in df.columns]
        print(df[display_cols].head(10).to_string(index=False))

## Step 11: Spatial Microenvironment Analysis

Calculate spatial relationships between cells and tumor/epithelia regions for microenvironment studies.

**New Columns Generated**:

1. **tumor_bed** (binary): Identifies tumor microenvironment regions
   - Value = 1 if cell is in tumor (tumor==1) OR in any tumor zone (zone_a_t, zone_b_t, zone_c_t, zone_d_t == 1)
   - Value = 0 otherwise (normal epithelia microenvironment)

2. **distance_tumor** (continuous): Signed distance to nearest tumor region
   - Positive values: Stromal cells approaching tumor (distance from stroma → tumor boundary)
   - Negative values: Tumor cells within tumor bed (distance from tumor → stroma boundary)
   - Only calculated within tumor_bed == 1 regions

3. **distance_epithelia** (continuous): Signed distance to nearest normal epithelia region
   - Positive values: Stromal cells approaching epithelia (distance from stroma → epithelia boundary)
   - Negative values: Epithelia cells within epithelia regions (distance from epithelia → stroma boundary)
   - Only calculated within tumor_bed == 0 regions

4. **bed_roi** (categorical): Identifies which specific tumor/epithelia clone each cell surrounds
   - Format: t_{number} for tumor bed regions (e.g., t_1, t_2)
   - Format: e_{number} for normal epithelia bed regions (e.g., e_1, e_2)
   - Based on closest tumor_roi or epithelia_roi

**Analysis Groups**: All calculations are performed separately for each (batch_id, tissue_roi) combination to ensure spatial accuracy.

**Use Cases**:
- Study tumor microenvironment composition at different distances from tumor boundary
- Compare normal epithelia microenvironments vs tumor microenvironments
- Track which specific tumor/epithelia clones cells are associated with
- Analyze gradient effects in stromal cells approaching different regions

In [ ]:
# =============================================================================
# STEP 11: SPATIAL MICROENVIRONMENT ANALYSIS
# =============================================================================
print("Step 11: Calculating spatial microenvironment metrics...")

from scipy.spatial.distance import cdist

# Load the unified CSV file with all phenotypes
UNIFIED_FILE = OUTPUT_DIR / 'all_samples_combined_corrected_phenotyped.csv'

if not UNIFIED_FILE.exists():
    print(f"Error: Unified file not found at {UNIFIED_FILE}")
    print("Please run Step 7 first to generate the unified CSV file.")
else:
    # Load data
    df = pd.read_csv(UNIFIED_FILE, low_memory=False)
    print(f"Loaded unified file: {len(df):,} rows")
    
    # Verify required columns exist
    required_cols = ['batch_id', 'tissue_roi', 'x_centroid', 'y_centroid', 
                     'tumor', 'epithelia', 'stroma', 'tumor_roi', 'epithelia_roi']
    zone_cols = ['zone_a_t', 'zone_b_t', 'zone_c_t', 'zone_d_t']
    
    missing_cols = [col for col in required_cols + zone_cols if col not in df.columns]
    if missing_cols:
        print(f"Error: Missing required columns: {missing_cols}")
    else:
        print(f"All required columns present. Starting spatial analysis...")
        
        # Initialize new columns
        df['tumor_bed'] = 0
        df['distance_tumor'] = np.nan
        df['distance_epithelia'] = np.nan
        df['bed_roi'] = ''
        
        # Step 1: Calculate tumor_bed column
        print("\nStep 1: Calculating tumor_bed column...")
        tumor_bed_mask = (df['tumor'] == 1) | \
                        (df['zone_a_t'] == 1) | \
                        (df['zone_b_t'] == 1) | \
                        (df['zone_c_t'] == 1) | \
                        (df['zone_d_t'] == 1)
        df.loc[tumor_bed_mask, 'tumor_bed'] = 1
        
        n_tumor_bed = (df['tumor_bed'] == 1).sum()
        n_normal_bed = (df['tumor_bed'] == 0).sum()
        print(f"  Tumor bed cells: {n_tumor_bed:,} ({n_tumor_bed/len(df)*100:.2f}%)")
        print(f"  Normal bed cells: {n_normal_bed:,} ({n_normal_bed/len(df)*100:.2f}%)")
        
        # Step 2: Process each (batch_id, tissue_roi) group separately
        print("\nStep 2: Calculating distances and bed_roi for each group...")
        
        groups = df.groupby(['batch_id', 'tissue_roi'])
        n_groups = len(groups)
        print(f"Processing {n_groups} groups (batch_id, tissue_roi combinations)...")
        
        processed_groups = 0
        for (batch_id, tissue_roi), group_df in groups:
            processed_groups += 1
            if processed_groups % 100 == 0:
                print(f"  Processed {processed_groups}/{n_groups} groups...")
            
            group_indices = group_df.index
            
            # Get cell coordinates
            coords = group_df[['x_centroid', 'y_centroid']].values
            
            # =================================================================
            # TUMOR BED PROCESSING (tumor_bed == 1)
            # =================================================================
            tumor_bed_idx = group_df.index[group_df['tumor_bed'] == 1].tolist()
            
            if tumor_bed_idx:
                # Find tumor and stroma cells within tumor bed
                tumor_cells_idx = group_df.index[(group_df['tumor_bed'] == 1) & (group_df['tumor'] == 1)].tolist()
                stroma_cells_idx = group_df.index[(group_df['tumor_bed'] == 1) & (group_df['stroma'] == 1)].tolist()
                
                # Get unique tumor ROIs in this group
                tumor_rois = group_df.loc[group_df['tumor_roi'] > 0, 'tumor_roi'].unique()
                
                if len(tumor_rois) > 0:
                    # Calculate distances for each cell to each tumor ROI
                    for cell_idx in tumor_bed_idx:
                        cell_coord = coords[group_df.index.get_loc(cell_idx)].reshape(1, -1)
                        
                        # Find closest tumor ROI
                        min_distance = np.inf
                        closest_roi = None
                        
                        for roi_id in tumor_rois:
                            roi_cells_idx = group_df.index[(group_df['tumor_roi'] == roi_id)].tolist()
                            if roi_cells_idx:
                                roi_coords = group_df.loc[roi_cells_idx, ['x_centroid', 'y_centroid']].values
                                distances = cdist(cell_coord, roi_coords, metric='euclidean').flatten()
                                roi_min_dist = distances.min()
                                
                                if roi_min_dist < min_distance:
                                    min_distance = roi_min_dist
                                    closest_roi = roi_id
                        
                        # Assign bed_roi
                        if closest_roi is not None:
                            df.at[cell_idx, 'bed_roi'] = f't_{int(closest_roi)}'
                        
                        # Assign distance_tumor with sign based on cell type
                        if cell_idx in tumor_cells_idx:
                            # Tumor cells get negative distance (within tumor)
                            df.at[cell_idx, 'distance_tumor'] = -min_distance
                        elif cell_idx in stroma_cells_idx:
                            # Stromal cells get positive distance (approaching tumor)
                            df.at[cell_idx, 'distance_tumor'] = min_distance
            
            # =================================================================
            # NORMAL EPITHELIA BED PROCESSING (tumor_bed == 0)
            # =================================================================
            normal_bed_idx = group_df.index[group_df['tumor_bed'] == 0].tolist()
            
            if normal_bed_idx:
                # Find epithelia and stroma cells within normal bed
                epi_cells_idx = group_df.index[(group_df['tumor_bed'] == 0) & (group_df['epithelia'] == 1)].tolist()
                stroma_cells_idx = group_df.index[(group_df['tumor_bed'] == 0) & (group_df['stroma'] == 1)].tolist()
                
                # Get unique epithelia ROIs in this group
                epi_rois = group_df.loc[group_df['epithelia_roi'] > 0, 'epithelia_roi'].unique()
                
                if len(epi_rois) > 0:
                    # Calculate distances for each cell to each epithelia ROI
                    for cell_idx in normal_bed_idx:
                        cell_coord = coords[group_df.index.get_loc(cell_idx)].reshape(1, -1)
                        
                        # Find closest epithelia ROI
                        min_distance = np.inf
                        closest_roi = None
                        
                        for roi_id in epi_rois:
                            roi_cells_idx = group_df.index[(group_df['epithelia_roi'] == roi_id)].tolist()
                            if roi_cells_idx:
                                roi_coords = group_df.loc[roi_cells_idx, ['x_centroid', 'y_centroid']].values
                                distances = cdist(cell_coord, roi_coords, metric='euclidean').flatten()
                                roi_min_dist = distances.min()
                                
                                if roi_min_dist < min_distance:
                                    min_distance = roi_min_dist
                                    closest_roi = roi_id
                        
                        # Assign bed_roi
                        if closest_roi is not None:
                            df.at[cell_idx, 'bed_roi'] = f'e_{int(closest_roi)}'
                        
                        # Assign distance_epithelia with sign based on cell type
                        if cell_idx in epi_cells_idx:
                            # Epithelia cells get negative distance (within epithelia)
                            df.at[cell_idx, 'distance_epithelia'] = -min_distance
                        elif cell_idx in stroma_cells_idx:
                            # Stromal cells get positive distance (approaching epithelia)
                            df.at[cell_idx, 'distance_epithelia'] = min_distance
        
        print(f"  Completed processing all {n_groups} groups")
        
        # Step 3: Summary statistics
        print(f"\n{'='*80}")
        print(f"SPATIAL MICROENVIRONMENT ANALYSIS SUMMARY")
        print(f"{'='*80}")
        
        # tumor_bed summary
        print(f"\nTumor Bed Distribution:")
        print(f"  Tumor microenvironment (tumor_bed=1): {n_tumor_bed:,} cells")
        print(f"  Normal epithelia microenvironment (tumor_bed=0): {n_normal_bed:,} cells")
        
        # distance_tumor summary
        n_distance_tumor = df['distance_tumor'].notna().sum()
        if n_distance_tumor > 0:
            tumor_pos = df['distance_tumor'] > 0
            tumor_neg = df['distance_tumor'] < 0
            print(f"\nDistance to Tumor (within tumor_bed=1):")
            print(f"  Cells with distance_tumor values: {n_distance_tumor:,}")
            print(f"  Positive (stroma→tumor): {tumor_pos.sum():,} cells")
            print(f"  Negative (tumor→stroma): {tumor_neg.sum():,} cells")
            if tumor_pos.sum() > 0:
                print(f"  Positive range: {df.loc[tumor_pos, 'distance_tumor'].min():.2f} to {df.loc[tumor_pos, 'distance_tumor'].max():.2f}")
            if tumor_neg.sum() > 0:
                print(f"  Negative range: {df.loc[tumor_neg, 'distance_tumor'].min():.2f} to {df.loc[tumor_neg, 'distance_tumor'].max():.2f}")
        
        # distance_epithelia summary
        n_distance_epi = df['distance_epithelia'].notna().sum()
        if n_distance_epi > 0:
            epi_pos = df['distance_epithelia'] > 0
            epi_neg = df['distance_epithelia'] < 0
            print(f"\nDistance to Epithelia (within tumor_bed=0):")
            print(f"  Cells with distance_epithelia values: {n_distance_epi:,}")
            print(f"  Positive (stroma→epithelia): {epi_pos.sum():,} cells")
            print(f"  Negative (epithelia→stroma): {epi_neg.sum():,} cells")
            if epi_pos.sum() > 0:
                print(f"  Positive range: {df.loc[epi_pos, 'distance_epithelia'].min():.2f} to {df.loc[epi_pos, 'distance_epithelia'].max():.2f}")
            if epi_neg.sum() > 0:
                print(f"  Negative range: {df.loc[epi_neg, 'distance_epithelia'].min():.2f} to {df.loc[epi_neg, 'distance_epithelia'].max():.2f}")
        
        # bed_roi summary
        n_bed_roi = (df['bed_roi'] != '').sum()
        tumor_bed_rois = df[df['bed_roi'].str.startswith('t_', na=False)]['bed_roi'].nunique()
        epi_bed_rois = df[df['bed_roi'].str.startswith('e_', na=False)]['bed_roi'].nunique()
        print(f"\nBed ROI Assignments:")
        print(f"  Cells assigned to bed_roi: {n_bed_roi:,}")
        print(f"  Unique tumor bed ROIs (t_*): {tumor_bed_rois}")
        print(f"  Unique epithelia bed ROIs (e_*): {epi_bed_rois}")
        
        # Save updated unified file
        df.to_csv(UNIFIED_FILE, index=False)
        print(f"\n{'='*80}")
        print(f"Updated unified file saved: {UNIFIED_FILE}")
        print(f"New columns added: tumor_bed, distance_tumor, distance_epithelia, bed_roi")
        print(f"{'='*80}")
        
        # Display example rows
        print(f"\nExample rows with new spatial columns:")
        display_cols = ['cell_id', 'batch_id', 'tissue_roi', 'tumor', 'epithelia', 'stroma',
                       'tumor_bed', 'distance_tumor', 'distance_epithelia', 'bed_roi']
        display_cols = [col for col in display_cols if col in df.columns]
        print(df[display_cols].head(20).to_string(index=False))